# 01 — Data Collection

This notebook collects raw data from external APIs and saves it to `../Data/`.
Run once — data is already saved in the repository.

**Sources:**
- ENTSO-E: hourly day-ahead electricity prices for Belgium (2022–2025)
- Open-Meteo: hourly weather data for Antwerp (2022–2025)
- demandlib: synthetic household consumption profile (Belgium, 2022–2025)

## 1. ENTSO-E — Day-Ahead Prices (Belgium)

In [ ]:
from entsoe import EntsoePandasClient
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("ENTSOE_API_KEY")

client = EntsoePandasClient(api_key=api_key)

start = pd.Timestamp('2022-01-01', tz='Europe/Brussels')
end   = pd.Timestamp('2025-12-31', tz='Europe/Brussels')

prices = client.query_day_ahead_prices('BE', start=start, end=end)

print(prices.head(10))
print(f"Period : {prices.index[0]} → {prices.index[-1]}")
print(f"Rows   : {len(prices)}")
print(f"Missing: {prices.isnull().sum()}")

In [ ]:
df_prices = prices.reset_index()
df_prices.columns = ['timestamp', 'price_eur_mwh']
df_prices['price_eur_kwh'] = df_prices['price_eur_mwh'] / 1000

df_prices.to_csv('../Data/prices_be.csv', index=False)
print('Saved: ../Data/prices_be.csv')

## 2. Open-Meteo — Historical Weather (Antwerp)

In [ ]:
import openmeteo_requests
import requests_cache
from retry_requests import retry

cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

params = {
    "latitude": 51.2194,
    "longitude": 4.4025,
    "start_date": "2022-01-01",
    "end_date": "2025-12-31",
    "hourly": ["temperature_2m", "wind_speed_10m", "shortwave_radiation", "cloud_cover"]
}

responses = openmeteo.weather_api("https://archive-api.open-meteo.com/v1/archive", params=params)
response = responses[0]
hourly = response.Hourly()

df_weather = pd.DataFrame({
    "timestamp"         : pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit='s', utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit='s', utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive='left'
    ),
    "temperature_2m"    : hourly.Variables(0).ValuesAsNumpy(),
    "wind_speed_10m"    : hourly.Variables(1).ValuesAsNumpy(),
    "shortwave_radiation": hourly.Variables(2).ValuesAsNumpy(),
    "cloud_cover"       : hourly.Variables(3).ValuesAsNumpy(),
})

df_weather.to_csv('../Data/weather_antwerp.csv', index=False)
print(f'Saved: ../Data/weather_antwerp.csv  ({len(df_weather)} rows)')

## 3. Household Consumption Profile (synthetic, demandlib)

In [ ]:
import demandlib.bdew as bdew
import pandas as pd

annual_demand = 3500
holidays = []

all_years = [2022, 2023, 2024, 2025]
frames = []

for year in all_years:
    e_slp = bdew.ElecSlp(year, holidays=holidays)
    demand = e_slp.get_scaled_power_profiles({"h0": annual_demand})
    demand_hourly = demand.resample("h").mean()
    frames.append(demand_hourly)

load_profile = pd.concat(frames)
load_profile.index = pd.to_datetime(load_profile.index)

load_profile.reset_index().rename(columns={'index': 'timestamp'}).to_csv('../Data/load_profile.csv', index=False)
print(f'Saved: ../Data/load_profile.csv  ({len(load_profile)} rows)')